# 2. Custom architectures

This notebook defines the two custom models, checks their outputs, derives every layer's parameter
count and counts the multiply-accumulates of one image. Nothing is trained here: Model B is trained
in notebook 03 and Model A in notebook 04.

**Writes:** `results/tables/custom_architectures.md`, `results/tables/param_breakdown.md`.

## Setup

Folders, the random seed, the device, and small helpers for files and figures. Every notebook in
this folder starts with the same setup.

In [ ]:
import json
import os
import platform
import random
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torchvision

# Everything is read from and written to folders next to this notebook.
DATA_DIR = Path("data")                # the downloaded dataset and a decoded-pixel cache
RESULTS_DIR = Path("results")          # metrics, figures and tables
CHECKPOINT_DIR = Path("checkpoints")   # trained weights

SEED = 42            # fixes the split, the initial weights, augmentation and batch order
NUM_CLASSES = 10     # EuroSAT land-cover classes
INPUT_SHAPE = (3, 64, 64)

# cuBLAS needs this workspace setting for repeatable matrix products on a GPU.
CUBLAS_WORKSPACE = ":4096:8"


def seed_everything(seed, deterministic=True):
    """Seed Python, NumPy and PyTorch, so a run gives the same numbers every time.

    With ``deterministic=True``, cuDNN uses deterministic algorithms and stops
    auto-tuning, and PyTorch prefers deterministic implementations. This costs a
    little speed. Without it, two runs of the same experiment can differ by a few
    tenths of a percent, the same size as some differences between the models.
    A few GPU operations have no deterministic version; PyTorch then warns
    instead of stopping.
    """
    if deterministic:
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", CUBLAS_WORKSPACE)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # seeds the CPU and every GPU
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def cpu_name():
    """The processor's model name, or an empty string if the operating system won't say."""
    name = ""
    try:
        if sys.platform == "win32":
            import winreg

            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0"
            )
            name = str(winreg.QueryValueEx(key, "ProcessorNameString")[0])
        elif sys.platform.startswith("linux"):
            with open("/proc/cpuinfo", encoding="utf-8") as handle:
                for line in handle:
                    if line.lower().startswith("model name"):
                        name = line.split(":", 1)[1]
                        break
        elif sys.platform == "darwin":
            name = subprocess.run(
                ["sysctl", "-n", "machdep.cpu.brand_string"],
                capture_output=True, text=True, timeout=5, check=False,
            ).stdout
    except Exception:
        name = ""
    name = name or platform.processor() or ""
    name = re.sub(r"\((R|TM|C)\)", "", name, flags=re.IGNORECASE)  # trademark signs
    return " ".join(name.split())


def describe_device(device):
    """A readable name for a device, such as ``cuda:0 (Tesla T4)``.

    Every timing is recorded together with this name: a time means little
    without the hardware it was measured on.
    """
    device = torch.device(device)
    if device.type == "cuda":
        index = device.index if device.index is not None else torch.cuda.current_device()
        return f"cuda:{index} ({torch.cuda.get_device_name(index)})"
    name = cpu_name()
    return f"cpu ({name})" if name else "cpu"


#: Model name -> function building it; filled in by the model definitions below.
MODEL_BUILDERS = {}


def build_model(name, num_classes=NUM_CLASSES):
    """Seed PyTorch, then build ``name``: the same seed always gives the same initial weights.

    The three optimizer runs rely on this to start from identical weights, so
    that their comparison measures the optimizer and not the initialisation.
    """
    torch.manual_seed(SEED)
    return MODEL_BUILDERS[name](num_classes)


def count_parameters(model, trainable_only=True):
    """Number of parameters; frozen ones are left out when ``trainable_only``."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def model_size_kb(model):
    """Size of the saved weights (``state_dict``) in KB, where 1 KB = 1,024 bytes.

    Saving the weights, as a device would store them, rather than counting 4
    bytes per parameter, also counts buffers such as batch-norm statistics.
    """
    import io

    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1024


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python       {platform.python_version()} ({platform.system()})")
print(f"PyTorch      {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"NumPy        {np.__version__}")
print(f"Device       {describe_device(DEVICE)}")

Figures are shown inline at a small size and saved at print quality. Each model and each optimizer
keeps one colour and one name in every figure and table.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 100,            # inline figures: small, so the notebook stays light
    "savefig.dpi": 200,           # saved figures: print quality
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 1.8,
})

MODEL_LABELS = {
    "model_a": "Model A",
    "model_b": "Model B",
    "mobilenet_v2": "MobileNetV2",
    "squeezenet1_1": "SqueezeNet 1.1",
}
MODEL_COLORS = {
    "model_a": "#4C72B0",
    "model_b": "#DD8452",
    "mobilenet_v2": "#55A868",
    "squeezenet1_1": "#C44E52",
}
OPTIMIZER_LABELS = {"sgd": "SGD", "sgd_momentum": "SGD + momentum", "adam": "Adam"}
OPTIMIZER_COLORS = {"sgd": "#8172B3", "sgd_momentum": "#937860", "adam": "#DA8BC3"}


def save_figure(fig, path):
    """Save a figure as a PNG, creating its folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")

Results are saved as JSON and Markdown, so the tables can be rebuilt without retraining.

In [ ]:
def write_json(path, payload):
    """Write ``payload`` as indented JSON, creating the folder if needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8", newline="\n")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


def format_cell(value, spec=None):
    """One table cell: thousands separators for whole numbers, ``spec`` for decimals."""
    if value is None:
        return "-"
    if isinstance(value, bool):
        return str(value)
    if spec is not None and isinstance(value, (int, float)):
        return format(value, spec)
    if isinstance(value, int):
        return f"{value:,}"
    if isinstance(value, float):
        return f"{value:,.1f}" if abs(value) >= 1000 else f"{value:.4g}"
    return str(value).replace("|", "\\|")


def markdown_table(headers, rows, caption=None, formats=None):
    """A Markdown table. Number columns are right-aligned; ``formats`` fixes their decimals."""
    formats = formats or [None] * len(headers)

    def numeric(column):
        cells = [row[column] for row in rows if row[column] is not None]
        return bool(cells) and all(
            isinstance(cell, (int, float)) and not isinstance(cell, bool) for cell in cells
        )

    lines = [f"*{caption}*", ""] if caption else []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("|" + "|".join("---:" if numeric(c) else "---" for c in range(len(headers))) + "|")
    for row in rows:
        lines.append("| " + " | ".join(format_cell(v, s) for v, s in zip(row, formats)) + " |")
    return "\n".join(lines) + "\n"

## The custom models

**Model A** is a standard CNN: three blocks of 3×3 convolution, batch normalisation, ReLU and 2×2 max
pooling, then a fully connected head on the flattened feature maps. It is not limited in size; it is
the reference point that shows what Model B saves and what it costs.

**Model B** stays under 100,000 trainable parameters through two separate mechanisms:

1. **Depthwise-separable convolutions.** A standard 3×3 convolution from `C_in` to `C_out` channels
   has `9 × C_in × C_out` weights. A depthwise 3×3 convolution (one filter per input channel,
   `9 × C_in`) followed by a pointwise 1×1 convolution (`C_in × C_out`) has `9 × C_in + C_in × C_out`,
   about `1/C_out + 1/9` of the standard cost in both parameters and multiply-accumulates.
2. **Global average pooling instead of flattening.** Model A's flatten into a 128-unit layer alone
   holds 8,192 × 128 weights; Model B averages each feature map to one number, so its classifier
   holds only 320 × 10 weights.

Model B uses ReLU6, a single `min(max(x, 0), 6)`: no exponential and no lookup table, so it costs
almost nothing on a microcontroller, and its bounded output range keeps 8-bit quantisation of the
activations well scaled. Convolutions followed by batch normalisation have no bias, because batch
normalisation re-centres the output straight away.

Both models return raw scores (logits); the softmax is part of the cross-entropy loss.

In [ ]:
from torch import nn

MODEL_A_SETTINGS = {
    "blocks": [  # each block: convolution, batch norm, ReLU, then 2x2 max pooling
        {"out_channels": 32, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
        {"out_channels": 64, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
        {"out_channels": 128, "kernel_size": 3, "stride": 1, "padding": 1, "pool": 2},
    ],
    "head": {"global_pool": False, "fc_units": [128], "dropout": 0.3},
    "activation": "relu",
    "batch_norm": True,
}

MODEL_B_SETTINGS = {
    "stem": {"out_channels": 16, "kernel_size": 3, "stride": 1, "padding": 1},
    "blocks": [  # each block: depthwise-separable convolution, then optional 2x2 max pooling
        {"out_channels": 32, "stride": 1, "pool": 2},
        {"out_channels": 64, "stride": 1, "pool": 2},
        {"out_channels": 160, "stride": 1, "pool": 2},
        {"out_channels": 320, "stride": 1},
    ],
    "head": {"global_pool": True, "fc_units": [], "dropout": 0.2},
    "activation": "relu6",
    "batch_norm": True,
}

MODEL_B_PARAMETER_BUDGET = 100_000


def get_activation(name):
    """The activation layer for a name in the settings."""
    activations = {"relu": nn.ReLU, "relu6": nn.ReLU6}
    return activations[name]()


def conv_bn_act(in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                activation="relu", batch_norm=True):
    """Convolution, batch normalisation, activation. ``k*k*in*out`` weights."""
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                        padding=padding, bias=not batch_norm)]
    if batch_norm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(get_activation(activation))
    return nn.Sequential(*layers)


def depthwise_separable_conv(in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                             activation="relu6", batch_norm=True):
    """Depthwise k x k convolution, then pointwise 1x1 convolution, each with batch norm and activation.

    ``groups=in_channels`` gives every input channel its own filter, so channels
    are only mixed by the 1x1 convolution. Cost: ``k*k*in + in*out`` weights,
    against ``k*k*in*out`` for a standard convolution.
    """
    layers = [nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding,
                        groups=in_channels, bias=not batch_norm)]                    # depthwise
    if batch_norm:
        layers.append(nn.BatchNorm2d(in_channels))
    layers.append(get_activation(activation))
    layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=not batch_norm))  # pointwise
    if batch_norm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(get_activation(activation))
    return nn.Sequential(*layers)


def build_head(layers, settings, input_shape, num_classes):
    """Add the classifier to a list of feature layers and return the whole model.

    The number of features entering the classifier is found by passing one
    blank image through the feature layers.
    """
    head = settings["head"]
    if head["global_pool"]:
        layers.append(nn.AdaptiveAvgPool2d(1))
    layers.append(nn.Flatten())

    features = nn.Sequential(*layers)
    features.eval()
    with torch.no_grad():
        in_features = features(torch.zeros(1, *input_shape)).shape[1]
    features.train()

    for units in head["fc_units"]:  # hidden layers: linear, activation, dropout
        layers.append(nn.Linear(in_features, units))
        layers.append(get_activation(settings["activation"]))
        layers.append(nn.Dropout(head["dropout"]))
        in_features = units
    if not head["fc_units"]:
        layers.append(nn.Dropout(head["dropout"]))  # no hidden layer: dropout before the output layer
    layers.append(nn.Linear(in_features, num_classes))  # raw scores, no softmax

    model = nn.Sequential(*layers)
    model.num_classes = num_classes
    return model


def build_model_a(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, settings=MODEL_A_SETTINGS):
    """Model A: [convolution, batch norm, ReLU, max pool] x 3, then a fully connected head."""
    layers = []
    channels = input_shape[0]
    for block in settings["blocks"]:
        layers.append(conv_bn_act(channels, block["out_channels"], block["kernel_size"],
                                  block["stride"], block["padding"],
                                  settings["activation"], settings["batch_norm"]))
        if block.get("pool"):
            layers.append(nn.MaxPool2d(block["pool"]))
        channels = block["out_channels"]
    return build_head(layers, settings, input_shape, num_classes)


def build_model_b(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, settings=MODEL_B_SETTINGS):
    """Model B: a standard 3x3 stem, depthwise-separable blocks, global average pooling."""
    stem = settings["stem"]
    layers = [conv_bn_act(input_shape[0], stem["out_channels"], stem["kernel_size"],
                          stem["stride"], stem["padding"],
                          settings["activation"], settings["batch_norm"])]
    channels = stem["out_channels"]
    for block in settings["blocks"]:
        layers.append(depthwise_separable_conv(channels, block["out_channels"],
                                               stride=block["stride"],
                                               activation=settings["activation"],
                                               batch_norm=settings["batch_norm"]))
        if block.get("pool"):
            layers.append(nn.MaxPool2d(block["pool"]))
        channels = block["out_channels"]
    return build_head(layers, settings, input_shape, num_classes)


MODEL_BUILDERS["model_a"] = build_model_a
MODEL_BUILDERS["model_b"] = build_model_b

## Counting multiply-accumulates

A multiply-accumulate (MAC) is one multiplication and one addition; one image's forward pass is
counted at batch size 1. Only convolutions and fully connected layers are counted, the convention
of the MobileNet and SqueezeNet papers and of torchvision's published figures (MobileNetV2 0.301 G
and SqueezeNet 1.1 0.349 G MACs at 224×224, which this count reproduces). Batch normalisation,
activations, pooling and bias additions are cheap next to the convolutions and are left out.

- a convolution costs (output values) × (input channels / groups) × (kernel area), so a depthwise
  convolution, with one group per channel, costs a fraction of a standard one;
- a fully connected layer costs (output values) × (input features).

In [ ]:
import copy
import math

from torch import nn


def count_macs(model, input_shape=INPUT_SHAPE):
    """Multiply-accumulates of convolutions and fully connected layers, for one image.

    Measured on a copy in evaluation mode, so the model itself is untouched.
    """
    probe = copy.deepcopy(model).cpu().eval()
    total = 0

    def conv(module, inputs, output):
        nonlocal total
        total += output[0].numel() * (module.in_channels // module.groups) * math.prod(module.kernel_size)

    def linear(module, inputs, output):
        nonlocal total
        total += output[0].numel() * module.in_features

    hooks = []
    for module in probe.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(conv))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear))
    try:
        with torch.no_grad():
            probe(torch.zeros(1, *input_shape))
    finally:
        for hook in hooks:
            hook.remove()
    return int(total)

## Layer-by-layer parameter tables

One row per layer: its type, kernel, stride, output shape and parameter count, with the formula
that gives the count. For a convolution that is kernel height × width × (input channels / groups)
× filters, plus one bias per filter when there is a bias. A batch-normalisation layer learns a scale
and a shift per channel; its running statistics are buffers, not trainable parameters. Output shapes
come from passing one 64×64 image through the model. The total is checked against the model's own
count, so the table cannot disagree with the model.

In [ ]:
import copy

import pandas as pd
from torch import nn


def layer_type(module):
    if isinstance(module, nn.Conv2d):
        if module.groups > 1 and module.groups == module.in_channels:
            return "Conv2d depthwise"
        if module.kernel_size == (1, 1):
            return "Conv2d pointwise"
        return "Conv2d"
    if isinstance(module, nn.AdaptiveAvgPool2d) and module.output_size in (1, (1, 1)):
        return "Global average pool"
    if isinstance(module, nn.Dropout):
        return f"Dropout (p={module.p})"
    return type(module).__name__


def kernel_of(module):
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
        size = module.kernel_size
        height, width = (size, size) if isinstance(size, int) else size
        return f"{height}×{width}"
    return ""


def stride_of(module):
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
        stride = module.stride
        return str(stride if isinstance(stride, int) else stride[0])
    return ""


def parameter_formula(module, params):
    """How ``params`` is calculated for this layer, such as ``3×3×3×16 = 432``."""
    if not params:
        return ""
    if isinstance(module, nn.Conv2d):
        height, width = module.kernel_size
        weights = f"{height}×{width}×{module.in_channels // module.groups}×{module.out_channels}"
        bias = f" + {module.out_channels}" if module.bias is not None else ""
        return f"{weights}{bias} = {params:,}"
    if isinstance(module, nn.BatchNorm2d):
        return f"2×{module.num_features} = {params:,}"
    if isinstance(module, nn.Linear):
        bias = f" + {module.out_features}" if module.bias is not None else ""
        return f"{module.in_features}×{module.out_features}{bias} = {params:,}"
    return f"{params:,}"


def layer_table(model, input_shape=INPUT_SHAPE):
    """Every layer with its output shape, parameters and their formula; the last row is the total."""
    probe = copy.deepcopy(model).cpu().eval()
    layers = [(name, module) for name, module in probe.named_modules()
              if name and next(module.children(), None) is None]
    shapes = {}

    def recorder(name):
        def record(module, inputs, output):
            shapes[name] = tuple(output.shape[1:])
        return record

    hooks = [module.register_forward_hook(recorder(name)) for name, module in layers]
    with torch.no_grad():
        probe(torch.zeros(1, *input_shape))
    for hook in hooks:
        hook.remove()

    rows = []
    for name, module in layers:
        params = sum(p.numel() for p in module.parameters(recurse=False) if p.requires_grad)
        rows.append({
            "layer": name, "type": layer_type(module), "kernel": kernel_of(module),
            "stride": stride_of(module), "output": "×".join(str(s) for s in shapes[name]),
            "params": params, "formula": parameter_formula(module, params),
        })
    total = sum(row["params"] for row in rows)
    if total != count_parameters(model):
        raise ValueError("some parameters are defined outside any layer")
    for row in rows:
        row["share"] = f"{100 * row['params'] / total:.1f}%" if row["params"] else ""
    rows.append({"layer": "total", "type": "", "kernel": "", "stride": "", "output": "",
                 "params": total, "formula": "", "share": "100%"})
    return pd.DataFrame(rows)


def first_activation(model):
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.ReLU6)):
            return type(module).__name__
    return "none"


def width_of(model, layer):
    """Filters of a convolution, or units of a fully connected layer."""
    module = dict(model.named_modules())[layer]
    if isinstance(module, nn.Conv2d):
        return str(module.out_channels)
    if isinstance(module, nn.Linear):
        return str(module.out_features)
    return ""


def architecture_tables(model_a, model_b, input_shape=INPUT_SHAPE):
    """The two Section 2 tables as Markdown, saved to ``results/tables/``.

    - ``custom_architectures.md``: an overview of both models, then each model's
      main layers with kernel size, filters or units, stride and output shape;
    - ``param_breakdown.md``: every layer of each model with its parameter formula.
    """
    models = {"Model A": model_a, "Model B": model_b}
    tables = {name: layer_table(model, input_shape) for name, model in models.items()}
    size = "×".join(str(value) for value in input_shape)

    overview = []
    for name, model in models.items():
        types = tables[name][tables[name]["layer"] != "total"]["type"].tolist()
        overview.append([name, types.count("Conv2d"), types.count("Conv2d depthwise"),
                         types.count("Conv2d pointwise"), types.count("Linear"),
                         first_activation(model), int(tables[name]["params"].iloc[-1]),
                         count_macs(model, input_shape)])
    sections = [
        "# Custom architectures\n",
        markdown_table(
            ["Model", "Standard conv", "Depthwise conv", "Pointwise conv", "Fully connected",
             "Activation", "Trainable parameters", "MACs per image"],
            overview,
            caption=f"Both models take a {size} image and return one raw score per class.",
        ),
    ]
    for name in models:
        main = tables[name][tables[name]["type"].str.startswith(("Conv2d", "Linear"))]
        sections.append(f"\n## {name} - main layers\n")
        sections.append(markdown_table(
            ["Layer", "Type", "Kernel", "Filters / units", "Stride", "Output", "Parameters"],
            [[row.layer, row.type, row.kernel, width_of(models[name], row.layer), row.stride,
              row.output, int(row.params)] for row in main.itertuples()],
        ))
    custom = "\n".join(sections)

    breakdown = ["# Parameter breakdown\n"]
    for name, table in tables.items():
        breakdown.append(f"\n## {name}\n")
        breakdown.append(markdown_table(
            ["Layer", "Type", "Kernel", "Stride", "Output", "Parameters", "Formula", "Share"],
            [[row.layer, row.type, row.kernel, row.stride, row.output, int(row.params),
              row.formula, row.share] for row in table.itertuples()],
        ))
    breakdown = "\n".join(breakdown)

    write_text(RESULTS_DIR / "tables" / "custom_architectures.md", custom)
    write_text(RESULTS_DIR / "tables" / "param_breakdown.md", breakdown)
    return custom, breakdown

## Build both models

In [ ]:
from IPython.display import Markdown, display

model_a = build_model("model_a")
model_b = build_model("model_b")
model_b

## Output check

Both models must return one raw score per class at any batch size. Rows summing to 1 would mean a
softmax had been left inside the model.

In [ ]:
for name, model in (("Model A", model_a), ("Model B", model_b)):
    model.eval()
    with torch.no_grad():
        for batch in (1, 2):
            scores = model(torch.randn(batch, *INPUT_SHAPE))
            softmax_inside = torch.allclose(scores.sum(dim=1), torch.ones(batch), atol=1e-3)
            print(f"{name}, batch of {batch}: output {tuple(scores.shape)}, softmax inside: {softmax_inside}")

## Model A, layer by layer

Most of Model A's parameters sit in one layer: the flattened 8×8×128 feature maps feed a
128-unit fully connected layer.

In [ ]:
layer_table(model_a)

## Model B, layer by layer

In [ ]:
layer_table(model_b)

## Parameters and multiply-accumulates

In [ ]:
for name, model in (("Model A", model_a), ("Model B", model_b)):
    print(f"{name}: {count_parameters(model):,} trainable parameters, "
          f"{count_macs(model):,} MACs per image")
assert count_parameters(model_b) <= MODEL_B_PARAMETER_BUDGET, "Model B is over its parameter budget"
print(f"Model B uses {count_parameters(model_b) / MODEL_B_PARAMETER_BUDGET:.0%} "
      f"of its {MODEL_B_PARAMETER_BUDGET:,}-parameter budget.")

## The Section 2 tables

In [ ]:
custom, breakdown = architecture_tables(model_a, model_b)
display(Markdown(custom))